In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import numpy as np

# ------------------------------------
# 1. 데이터 로드
# ------------------------------------
df = pd.read_csv(
    "/Users/mac/Desktop/project/company_data/third_week/data_csv/전국일반음식점.csv",
    encoding="CP949",
    low_memory=False
)

# ------------------------------------
# 2. 날짜 처리
# ------------------------------------
df["인허가일자"] = pd.to_datetime(df["인허가일자"], errors="coerce")
df["폐업일자"] = pd.to_datetime(df["폐업일자"], errors="coerce")

# 최근 5년만 유지
df = df[df["인허가일자"] >= "2019-01-01"]

# ------------------------------------
# 3. 폐업 여부 생성
# ------------------------------------
df["폐업여부"] = df["폐업일자"].notnull().astype(int)

# ------------------------------------
# 4. 연도 생성
# ------------------------------------
df["year"] = df["인허가일자"].dt.year

# ------------------------------------
# 5. 업태별 연도별 성장(Net Growth)
# ------------------------------------
annual = (
    df.groupby(["위생업태명", "year"])
      .agg(
          신규=("인허가일자", "count"),
          폐업=("폐업여부", "sum")
      )
      .reset_index()
)
annual["net_growth"] = annual["신규"] - annual["폐업"]

df = df.merge(
    annual[["위생업태명", "year", "net_growth"]],
    on=["위생업태명", "year"],
    how="left"
)

# ------------------------------------
# 6. 영업기간 계산
# ------------------------------------
today = pd.Timestamp("2025-01-01")
df["영업종료일"] = df["폐업일자"].fillna(today)
df["영업기간"] = (df["영업종료일"] - df["인허가일자"]).dt.days.clip(lower=0)

# ------------------------------------
# 7. 면적 처리
# ------------------------------------
df["소재지면적"] = pd.to_numeric(df["소재지면적"], errors="coerce")
df["소재지면적"] = df["소재지면적"].fillna(df["소재지면적"].median())
df["log_면적"] = np.log1p(df["소재지면적"])

# ------------------------------------
# 8. 지역(시군구) 생성
# ------------------------------------
def extract_region(addr):
    try:
        parts = addr.split()
        if len(parts) >= 2:
            return parts[0] + " " + parts[1]
        return np.nan
    except:
        return np.nan

df["지역"] = df["소재지전체주소"].astype(str).apply(extract_region)

df = pd.get_dummies(df, columns=["지역"], drop_first=True)

# ------------------------------------
# 9. 업태명 더미 생성
# ------------------------------------
df = pd.get_dummies(df, columns=["위생업태명"], drop_first=True)

# ------------------------------------
# 10. X, y 구성
# ------------------------------------
업태_cols = [c for c in df.columns if c.startswith("위생업태명_")]
지역_cols = [c for c in df.columns if c.startswith("지역_")]

X = df[["영업기간", "net_growth", "log_면적"] + 업태_cols + 지역_cols]
y = df["폐업여부"]

X = X.fillna(0)

# ------------------------------------
# 11. Train/Test 분리
# ------------------------------------
X_train, X_test, Y_train, Y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ------------------------------------
# 12. 모델 학습
# ------------------------------------
model = LogisticRegression(max_iter=10000)
model.fit(X_train, Y_train)

# ------------------------------------
# 13. 평가
# ------------------------------------
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(Y_test, pred))
print("ROC-AUC  :", roc_auc_score(Y_test, prob))
print("\nConfusion Matrix:\n", confusion_matrix(Y_test, pred))
print("\nClassification Report:\n", classification_report(Y_test, pred))

# ------------------------------------
# 14. 회귀 계수
# ------------------------------------
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coef": model.coef_[0]
}).sort_values("coef", ascending=False)

print("\n--- 회귀 계수 정리 ---")
print(coef_df)

Accuracy : 0.6992719103238176
ROC-AUC  : 0.7139527997206757

Confusion Matrix:
 [[44810  7387]
 [18180 14640]]

Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.86      0.78     52197
           1       0.66      0.45      0.53     32820

    accuracy                           0.70     85017
   macro avg       0.69      0.65      0.66     85017
weighted avg       0.69      0.70      0.68     85017


--- 회귀 계수 정리 ---
            feature      coef
24         위생업태명_한식  5.823259
4          위생업태명_기타  4.036920
25      위생업태명_호프/통닭  1.401759
3         위생업태명_경양식  1.168956
31   지역_강원특별자치도 양구군  1.105254
..              ...       ...
154    지역_서울특별시 종로구 -0.637315
213     지역_전라남도 여수시 -0.656473
185  지역_세종특별자치시 해밀동 -0.724841
40   지역_강원특별자치도 평창군 -0.763053
33   지역_강원특별자치도 영월군 -0.916911

[251 rows x 2 columns]


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [5]:
# coef_df 가 feature / coef 로 구성된 DataFrame이라고 가정

# 1) 강원도 지역 변수만 필터
gw_coef = coef_df[coef_df["feature"].str.contains("지역_강원")]

# 2) 그 중 양의 계수(폐업 위험 증가 요인)만 추출
gw_positive = gw_coef[gw_coef["coef"] > 0]

# 3) 영향력 큰 순으로 정렬 (계수 큰 순)
gw_positive_sorted = gw_positive.sort_values("coef", ascending=False)

print("🔼 강원도 지역 중 폐업 위험 증가 요인(양의 계수)")
print(gw_positive_sorted)

🔼 강원도 지역 중 폐업 위험 증가 요인(양의 계수)
           feature      coef
31  지역_강원특별자치도 양구군  1.105254
39  지역_강원특별자치도 태백시  0.639783
28  지역_강원특별자치도 동해시  0.483814
37  지역_강원특별자치도 철원군  0.464602
42  지역_강원특별자치도 화천군  0.304089
29  지역_강원특별자치도 삼척시  0.281056
32  지역_강원특별자치도 양양군  0.059623


In [4]:
import pandas as pd

# coef_df가 이미 생성되어 있다고 가정
# (feature / coef 로 구성된 DataFrame)

# 1) 강원도 지역 변수만 필터
gw_coef = coef_df[coef_df["feature"].str.contains("지역_강원")]

# 2) 그 중에서 음의 계수(폐업 위험 감소 요인)만 추출
gw_negative = gw_coef[gw_coef["coef"] < 0]

# 3) 계수 절댓값 기준으로 정렬 (가장 영향력 큰 감소요인부터)
gw_negative_sorted = gw_negative.sort_values("coef", ascending=True)

print("🔽 강원도 지역 중 폐업 위험 감소 요인(음의 계수) TOP 전체")
print(gw_negative_sorted)

🔽 강원도 지역 중 폐업 위험 감소 요인(음의 계수) TOP 전체
           feature      coef
33  지역_강원특별자치도 영월군 -0.916911
40  지역_강원특별자치도 평창군 -0.763053
41  지역_강원특별자치도 홍천군 -0.532862
36  지역_강원특별자치도 정선군 -0.349065
35  지역_강원특별자치도 인제군 -0.214206
43  지역_강원특별자치도 횡성군 -0.114051
38  지역_강원특별자치도 춘천시 -0.101008
34  지역_강원특별자치도 원주시 -0.097999
30  지역_강원특별자치도 속초시 -0.057666
27  지역_강원특별자치도 고성군 -0.004015


In [6]:
# 강원도 관련 변수만 필터
gw_coef = coef_df[coef_df["feature"].str.contains("강원")]

# 양의 계수 = 폐업 위험 증가 요인
gw_up = gw_coef[gw_coef["coef"] > 0].sort_values("coef", ascending=False)

# 음의 계수 = 폐업 위험 감소 요인
gw_down = gw_coef[gw_coef["coef"] < 0].sort_values("coef", ascending=True)

print("강원도 폐업 위험 증가 요인")
print(gw_up)

print("\n강원도 폐업 위험 감소 요인")
print(gw_down)

강원도 폐업 위험 증가 요인
           feature      coef
31  지역_강원특별자치도 양구군  1.105254
39  지역_강원특별자치도 태백시  0.639783
28  지역_강원특별자치도 동해시  0.483814
37  지역_강원특별자치도 철원군  0.464602
42  지역_강원특별자치도 화천군  0.304089
29  지역_강원특별자치도 삼척시  0.281056
32  지역_강원특별자치도 양양군  0.059623

강원도 폐업 위험 감소 요인
           feature      coef
33  지역_강원특별자치도 영월군 -0.916911
40  지역_강원특별자치도 평창군 -0.763053
41  지역_강원특별자치도 홍천군 -0.532862
36  지역_강원특별자치도 정선군 -0.349065
35  지역_강원특별자치도 인제군 -0.214206
43  지역_강원특별자치도 횡성군 -0.114051
38  지역_강원특별자치도 춘천시 -0.101008
34  지역_강원특별자치도 원주시 -0.097999
30  지역_강원특별자치도 속초시 -0.057666
27  지역_강원특별자치도 고성군 -0.004015
